# 02 — Churn Drivers and Customer Segmentation

**Business question:** What characteristics distinguish customers who leave, and can we create an explainable retention-risk segment?

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.features import add_business_features

path = ROOT/"data/processed/demo_customers.csv"
if not path.exists():
    exec((ROOT/"data/generate_demo_data.py").read_text())
    main()
df = add_business_features(pd.read_csv(path))

In [ ]:
def risk_score(row):
    score = 0
    contract = row.get("contract_type", row.get("contract", ""))
    score += 3 if contract == "Month-to-month" else 0
    score += 2 if row["tenure"] < 12 else 0
    score += 1 if row.get("tech_support", row.get("techsupport", "")) == "No" else 0
    score += 1 if row["monthly_charges"] >= 85 else 0
    score += 1 if row.get("payment_method", "") == "Electronic check" else 0
    return score

df["risk_score"] = df.apply(risk_score, axis=1)
df["risk_segment"] = pd.cut(
    df["risk_score"], bins=[-1,2,5,99], labels=["Low","Medium","High"]
)

segment_table = (
    df.groupby("risk_segment", observed=True)
      .agg(customers=("customer_id","count"),
           churn_rate=("churn_flag","mean"),
           avg_monthly_charges=("monthly_charges","mean"))
)
segment_table

In [ ]:
numeric = [c for c in ["tenure","monthly_charges","total_charges","number_of_services","churn_flag"] if c in df]
corr = df[numeric].corr(numeric_only=True)
corr

In [ ]:
segment_table["churn_rate"].plot(kind="bar", title="Observed churn by explainable risk segment")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()

## Analyst takeaway

Compare the simple explainable score with later machine-learning risk estimates.
An analyst should be able to explain *why* a customer is considered risky, not only produce a score.